In [1]:
import models.voicecraft as voicecraft
import audiocraft

import torch
import torchaudio
import os
import numpy as np
import random

c:\Users\Sukiennik\miniconda3\envs\voicecraft\lib\site-packages\torchmetrics\utilities\imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
c:\Users\Sukiennik\miniconda3\envs\voicecraft\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'


In [ ]:
import os
os.chdir('/kaggle/working/VoiceCraft')

# Replace the torchmetrics import in voicecraft.py with a simple inline version
with open('models/voicecraft.py', 'r') as f:
    code = f.read()

code = code.replace(
    'from torchmetrics.classification import MulticlassAccuracy',
    '# torchmetrics removed - inline replacement\n'
    'class MulticlassAccuracy:\n'
    '    def __init__(self, **kwargs): pass\n'
    '    def __call__(self, *args, **kwargs): return 0.0\n'
    '    def to(self, *args, **kwargs): return self\n'
    '    def update(self, *args, **kwargs): pass\n'
    '    def compute(self, *args, **kwargs): return torch.tensor(0.0)\n'
    '    def reset(self, *args, **kwargs): pass'
)

with open('models/voicecraft.py', 'w') as f:
    f.write(code)

print("✅ Patched voicecraft.py")

import sys
sys.path.insert(0, '/kaggle/working/VoiceCraft')
from models import voicecraft
print("✅ voicecraft loaded!")

In [2]:
# Setting up tokenizers
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    print(f"Found {n_gpus} GPUs")
elif n_gpus == 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("Found 1 GPU")
else:
    print("No GPU, using CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# os.environ["USER"] = "user"
os.environ["USER"] = "Sukiennik" 

# --- HEBREW PHONEMIZER (replaces espeak TextTokenizer) ---
def hebrew_phonemize(text):
    letter_map = {
        'א': 'ʔ', 'ב': 'v', 'ג': 'g', 'ד': 'd', 'ה': 'h',
        'ו': 'v', 'ז': 'z', 'ח': 'χ', 'ט': 't', 'י': 'j',
        'כ': 'χ', 'ך': 'χ', 'ל': 'l', 'מ': 'm', 'ם': 'm',
        'נ': 'n', 'ן': 'n', 'ס': 's', 'ע': 'ʕ', 'פ': 'f',
        'ף': 'f', 'צ': 'ts', 'ץ': 'ts', 'ק': 'k', 'ר': 'ʁ',
        'ש': 'ʃ', 'ת': 't',
    }
    result = []
    for word in text.split():
        phones = []
        for char in word:
            if char in letter_map:
                phones.append(letter_map[char])
        if phones:
            result.append(' '.join(phones))
    return ' '.join(result)

print(f"Hebrew phonemizer test: שלום עולם -> {hebrew_phonemize('שלום עולם')}")

# --- AUDIO TOKENIZER (replaces audiocraft-based one) ---
from encodec.model import EncodecModel as EncodecModelPip
from encodec.quantization.vq import ResidualVectorQuantizer
from encodec.modules import SEANetEncoder, SEANetDecoder

class HebrewAudioTokenizer:
    """Drop-in replacement for AudioTokenizer using encodec pip package"""
    def __init__(self, signature):
        ckpt = torch.load(signature, map_location="cpu", weights_only=False)
        encoder = SEANetEncoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
            ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
            causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2)
        decoder = SEANetDecoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
            ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
            causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2, trim_right_ratio=1.0)
        quantizer = ResidualVectorQuantizer(dimension=128, n_q=4, bins=2048, decay=0.99,
            kmeans_init=True, kmeans_iters=50, threshold_ema_dead_code=2)
        self.model = EncodecModelPip(encoder=encoder, decoder=decoder, quantizer=quantizer,
            target_bandwidths=[1.5,3.0,6.0,12.0], sample_rate=16000, channels=1)
        self.model.load_state_dict(ckpt['best_state']['model'])
        self.model.eval()
        self.model.to(device)
        print("Audio tokenizer loaded")

    def encode(self, audio_tensor):
        with torch.no_grad():
            encoded = self.model.encode(audio_tensor.to(device))
            return encoded[0][0]  # codes: (batch, n_codebooks, n_frames)

    def decode(self, codes):
        with torch.no_grad():
            return self.model.decode([(codes, None)])

# --- LOAD VOICECRAFT MODEL ---
from models import voicecraft

print("✅ Ready for Hebrew speech editing!")

Found 1 GPU
Hebrew phonemizer test: שלום עולם -> ʃ l v m ʕ v l m
✅ Ready for Hebrew speech editing!


In [3]:
import shutil, soundfile as sf
from datasets import load_dataset
from IPython.display import Audio, display

left_margin = 0.08
right_margin = 0.08
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.8
temperature = 1
kvcache = 0
seed = 1
silence_tokens = [1388, 1898, 131]
stop_repetition = -1

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(seed)

print("Downloading Hebrew audio...")
ds = load_dataset("google/fleurs", "he_il", split="train", trust_remote_code=True)

for i in range(len(ds)):
    sample = ds[i]
    audio = np.array(sample["audio"]["array"], dtype=np.float32)
    sr = sample["audio"]["sampling_rate"]
    duration = len(audio) / sr
    if 5.0 < duration < 10.0:
        break

orig_transcript = sample["transcription"]
print(f"Sample {i}: '{orig_transcript}' ({duration:.1f}s)")

temp_folder = "./demo/temp_hebrew"
os.makedirs(temp_folder, exist_ok=True)
audio_fn = os.path.join(temp_folder, "hebrew_sample.wav")
if sr != 16000:
    import librosa
    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
sf.write(audio_fn, audio, 16000)

words = orig_transcript.split()
word_duration = duration / len(words)
align_fn = os.path.join(temp_folder, "hebrew_sample.csv")
with open(align_fn, "w") as f:
    f.write("Begin,End,Label,Type\n")
    for j, word in enumerate(words):
        begin = j * word_duration
        end = (j + 1) * word_duration
        f.write(f"{begin:.3f},{end:.3f},{word},words\n")

print(f"Transcript: {orig_transcript}")
print(f"Words: {len(words)}")
print("\nOriginal audio:")
display(Audio(audio, rate=16000))
print("\n✅ Cell B done")

Generating train split: 3242 examples [00:15, 204.78 examples/s]
Generating validation split: 328 examples [00:01, 230.36 examples/s]
Generating test split: 792 examples [00:03, 228.83 examples/s]


Sample 2: 'בסוף 2015 הקימה טוגי-נט את אסטרו-נט רדיו כתחנת בת' (6.5s)
Transcript: בסוף 2015 הקימה טוגי-נט את אסטרו-נט רדיו כתחנת בת
Words: 9

Original audio:



✅ Cell B done


In [4]:
import platform

# Fix for Windows: Monkey-patching os.uname (English comment)
if platform.system() == 'Windows':
    from collections import namedtuple
    def uname_windows():
        uname_result = namedtuple('uname_result', ['sysname', 'nodename', 'release', 'version', 'machine'])
        return uname_result(platform.system(), platform.node(), platform.release(), platform.version(), platform.machine())
    
    # Adding the missing attribute to the os module
    os.uname = uname_windows

print("✅ Windows compatibility patch applied.")

✅ Windows compatibility patch applied.


In [5]:
# Fix the dummy MulticlassAccuracy to accept any arguments
import models.voicecraft as vc_module

class MulticlassAccuracy:
    def __init__(self, *args, **kwargs): pass
    def __call__(self, *args, **kwargs): return 0.0
    def to(self, *args, **kwargs): return self
    def update(self, *args, **kwargs): pass
    def compute(self, *args, **kwargs): return torch.tensor(0.0)
    def reset(self, *args, **kwargs): pass

vc_module.MulticlassAccuracy = MulticlassAccuracy
print("✅ Fixed")

✅ Fixed


In [6]:
import torch.nn as nn
import models.voicecraft as vc_module

class MulticlassAccuracy(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
    def forward(self, *args, **kwargs):
        return torch.tensor(0.0)
    def update(self, *args, **kwargs): pass
    def compute(self, *args, **kwargs): return torch.tensor(0.0)
    def reset(self, *args, **kwargs): pass

vc_module.MulticlassAccuracy = MulticlassAccuracy
print("✅ Fixed")

✅ Fixed


In [ ]:
with open('/kaggle/working/VoiceCraft/inference_speech_editing_scale.py', 'r') as f:
    print(f.read())

In [7]:
import requests, time, logging

# Download models
voicecraft_name = "giga330M.pth"
ckpt_fn = f"./pretrained_models/{voicecraft_name}"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

def download_file(url, dest):
    if not os.path.exists(dest):
        print(f"Downloading {os.path.basename(dest)}...")
        r = requests.get(url, stream=True)
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Done")

os.makedirs("./pretrained_models", exist_ok=True)
download_file(f"https://huggingface.co/pyp1/VoiceCraft/resolve/main/{voicecraft_name}", ckpt_fn)
download_file("https://huggingface.co/pyp1/VoiceCraft/resolve/main/encodec_4cb2048_giga.th", encodec_fn)

# Load VoiceCraft model
print("Loading VoiceCraft model...")
ckpt = torch.load(ckpt_fn, map_location="cpu", weights_only=False)
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']
model_args = ckpt["config"]
print(f"Model loaded. Vocab size: {len(phn2num)}")

# Load Encodec using our working method
print("Loading Encodec...")
enc_ckpt = torch.load(encodec_fn, map_location="cpu", weights_only=False)
encoder = SEANetEncoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2)
decoder = SEANetDecoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2, trim_right_ratio=1.0)
quantizer = ResidualVectorQuantizer(dimension=128, n_q=4, bins=2048, decay=0.99,
    kmeans_init=True, kmeans_iters=50, threshold_ema_dead_code=2)
encodec_model = EncodecModelPip(encoder=encoder, decoder=decoder, quantizer=quantizer,
    target_bandwidths=[1.5,3.0,6.0,12.0], sample_rate=16000, channels=1)
encodec_model.load_state_dict(enc_ckpt['best_state']['model'])
encodec_model = encodec_model.to(device).eval()
print("Encodec loaded")

# --- SPEECH EDITING ---
# What we're changing
target_transcript = "בסוף 2015 הקימה טוגי-נט את אסטרו-נט טלוויזיה כתחנת בת"
print(f"\nOriginal:  {orig_transcript}")
print(f"Target:    {target_transcript}")
print(f"Change:    רדיו -> טלוויזיה")

# Phonemize target transcript
target_phones = hebrew_phonemize(target_transcript)
print(f"Phonemes:  {target_phones}")

# Convert phonemes to token IDs (map what we can, skip unknown)
phone_list = target_phones.split()
text_tokens = []
mapped = 0
for phn in phone_list:
    if phn in phn2num:
        text_tokens.append(phn2num[phn])
        mapped += 1
print(f"Mapped {mapped}/{len(phone_list)} phonemes to model vocabulary")

text_tokens = torch.LongTensor(text_tokens).unsqueeze(0)
text_tokens_lens = torch.LongTensor([text_tokens.shape[-1]])

# Encode original audio with our Encodec
audio_wav, sr_loaded = torchaudio.load(audio_fn)
if sr_loaded != 16000:
    audio_wav = torchaudio.transforms.Resample(sr_loaded, 16000)(audio_wav)
audio_wav = audio_wav.unsqueeze(0).to(device)  # [1, 1, T]

with torch.no_grad():
    encoded = encodec_model.encode(audio_wav)
    original_codes = encoded[0][0]  # [1, 4, T]

original_audio = original_codes.transpose(1, 2)  # [1, T, 4]
print(f"Audio encoded: {original_audio.shape[1]} frames = {original_audio.shape[1]/codec_sr:.1f}s")

# Create mask interval
import csv
alignments = []
with open(align_fn, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        alignments.append(row)

# Find the word we're changing (word index 6 = "רדיו")
orig_words = orig_transcript.split()
target_words = target_transcript.split()
edit_idx = None
for idx in range(len(orig_words)):
    if idx >= len(target_words) or orig_words[idx] != target_words[idx]:
        edit_idx = idx
        break

start = float(alignments[edit_idx]['Begin'])
end = float(alignments[edit_idx]['End'])
morphed_span = (max(start - left_margin, 1/codec_sr), min(end + right_margin, duration))
mask_interval = [[round(morphed_span[0]*codec_sr), round(morphed_span[1]*codec_sr)]]
mask_interval = torch.LongTensor(mask_interval)
print(f"Masking word '{orig_words[edit_idx]}' at {start:.2f}s-{end:.2f}s")

# Run VoiceCraft inference
print("\nGenerating edited audio...")
stime = time.time()
with torch.no_grad():
    encoded_frames = model.inference(
        text_tokens.to(device),
        text_tokens_lens.to(device),
        original_audio[...,:model_args.n_codebooks].to(device),
        mask_interval=mask_interval.unsqueeze(0).to(device),
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        stop_repetition=stop_repetition,
        kvcache=kvcache,
        silence_tokens=silence_tokens,
    )
print(f"Generation took {time.time()-stime:.1f}s")

if type(encoded_frames) == tuple:
    encoded_frames = encoded_frames[0]

# Decode both original and generated audio
with torch.no_grad():
    original_sample = encodec_model.decode([(original_codes, None)])
    generated_sample = encodec_model.decode([(encoded_frames, None)])

orig_out = original_sample[0].cpu()
new_out = generated_sample[0].cpu()

from IPython.display import Audio, display
print("\nOriginal Audio:")
display(Audio(orig_out.squeeze().numpy(), rate=16000))
print("\nEdited Audio (רדיו -> טלוויזיה):")
display(Audio(new_out.squeeze().numpy(), rate=16000))
print("\n✅ Hebrew speech editing complete!")

Loading VoiceCraft model...
Model loaded. Vocab size: 80
Loading Encodec...


c:\Users\Sukiennik\miniconda3\envs\voicecraft\lib\site-packages\torch\nn\utils\weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Encodec loaded

Original:  בסוף 2015 הקימה טוגי-נט את אסטרו-נט רדיו כתחנת בת
Target:    בסוף 2015 הקימה טוגי-נט את אסטרו-נט טלוויזיה כתחנת בת
Change:    רדיו -> טלוויזיה
Phonemes:  v s v f h k j m h t v g j n t ʔ t ʔ s t ʁ v n t t l v v j z j h χ t χ n t v t
Mapped 35/39 phonemes to model vocabulary
Audio encoded: 327 frames = 6.5s
Masking word 'רדיו' at 4.36s-5.09s

Generating edited audio...
Generation took 22.2s

Original Audio:



Edited Audio (רדיו -> טלוויזיה):



✅ Hebrew speech editing complete!


In [8]:
# Pick sample 1 - longer and clearer
sample = ds[1]
audio = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]
duration = len(audio) / sr
orig_transcript = sample["transcription"]

print(f"Sample: '{orig_transcript}' ({duration:.1f}s)")

# Save audio
import soundfile as sf
if sr != 16000:
    import librosa
    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
audio_fn = "./demo/temp_hebrew/hebrew_sample.wav"
sf.write(audio_fn, audio, 16000)

# Create alignment
words = orig_transcript.split()
word_duration = duration / len(words)
align_fn = "./demo/temp_hebrew/hebrew_sample.csv"
with open(align_fn, "w") as f:
    f.write("Begin,End,Label,Type\n")
    for j, word in enumerate(words):
        begin = j * word_duration
        end = (j + 1) * word_duration
        f.write(f"{begin:.3f},{end:.3f},{word},words\n")

print(f"Words: {words}")
print(f"\nWhich word do you want to change? (pick a number 0-{len(words)-1})")
for j, w in enumerate(words):
    print(f"  {j}: {w}")

from IPython.display import Audio, display
print("\nAudio:")
display(Audio(audio, rate=16000))

Sample: 'כמו כן מטייל בריטי בספרד עלול להבין בטעות נפנוף לשלום בכף יד הפונה לכיוון המנופף ולא לכיוון האדם שאליו מנופפים כמחווה לחזור בחזרה' (13.6s)
Words: ['כמו', 'כן', 'מטייל', 'בריטי', 'בספרד', 'עלול', 'להבין', 'בטעות', 'נפנוף', 'לשלום', 'בכף', 'יד', 'הפונה', 'לכיוון', 'המנופף', 'ולא', 'לכיוון', 'האדם', 'שאליו', 'מנופפים', 'כמחווה', 'לחזור', 'בחזרה']

Which word do you want to change? (pick a number 0-22)
  0: כמו
  1: כן
  2: מטייל
  3: בריטי
  4: בספרד
  5: עלול
  6: להבין
  7: בטעות
  8: נפנוף
  9: לשלום
  10: בכף
  11: יד
  12: הפונה
  13: לכיוון
  14: המנופף
  15: ולא
  16: לכיוון
  17: האדם
  18: שאליו
  19: מנופפים
  20: כמחווה
  21: לחזור
  22: בחזרה

Audio:


In [9]:
import csv, time

# Edit setup
edit_idx = 11
new_word = "רגל"
target_words = orig_transcript.split()
target_words[edit_idx] = new_word
target_transcript = " ".join(target_words)

print(f"Original:  {orig_transcript}")
print(f"Target:    {target_transcript}")
print(f"Change:    יד -> רגל")

# Phonemize
target_phones = hebrew_phonemize(target_transcript)
print(f"Phonemes:  {target_phones}")

# Tokens
phone_list = target_phones.split()
text_tokens = [phn2num[p] for p in phone_list if p in phn2num]
text_tokens = torch.LongTensor(text_tokens).unsqueeze(0)
text_tokens_lens = torch.LongTensor([text_tokens.shape[-1]])

# Encode audio
audio_wav, sr_loaded = torchaudio.load(audio_fn)
if sr_loaded != 16000:
    audio_wav = torchaudio.transforms.Resample(sr_loaded, 16000)(audio_wav)
audio_wav = audio_wav.unsqueeze(0).to(device)

with torch.no_grad():
    encoded = encodec_model.encode(audio_wav)
    original_codes = encoded[0][0]

original_audio_tensor = original_codes.transpose(1, 2)

# Mask interval
alignments = []
with open(align_fn, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        alignments.append(row)

start = float(alignments[edit_idx]['Begin'])
end = float(alignments[edit_idx]['End'])
morphed_span = (max(start - left_margin, 1/codec_sr), min(end + right_margin, duration))
mask_interval = [[round(morphed_span[0]*codec_sr), round(morphed_span[1]*codec_sr)]]
mask_interval = torch.LongTensor(mask_interval)
print(f"Masking: {start:.2f}s - {end:.2f}s")

# Generate
print("\nGenerating...")
stime = time.time()
with torch.no_grad():
    encoded_frames = model.inference(
        text_tokens.to(device),
        text_tokens_lens.to(device),
        original_audio_tensor[...,:model_args.n_codebooks].to(device),
        mask_interval=mask_interval.unsqueeze(0).to(device),
        top_k=top_k, top_p=top_p, temperature=temperature,
        stop_repetition=stop_repetition, kvcache=kvcache,
        silence_tokens=silence_tokens,
    )
print(f"Done in {time.time()-stime:.1f}s")

if type(encoded_frames) == tuple:
    encoded_frames = encoded_frames[0]

with torch.no_grad():
    original_sample = encodec_model.decode([(original_codes, None)])
    generated_sample = encodec_model.decode([(encoded_frames, None)])

from IPython.display import Audio, display
print("\nOriginal:")
display(Audio(original_sample[0].squeeze().cpu().numpy(), rate=16000))
print("\nEdited (יד -> רגל):")
display(Audio(generated_sample[0].squeeze().cpu().numpy(), rate=16000))
print("\n✅ Done!")

Original:  כמו כן מטייל בריטי בספרד עלול להבין בטעות נפנוף לשלום בכף יד הפונה לכיוון המנופף ולא לכיוון האדם שאליו מנופפים כמחווה לחזור בחזרה
Target:    כמו כן מטייל בריטי בספרד עלול להבין בטעות נפנוף לשלום בכף רגל הפונה לכיוון המנופף ולא לכיוון האדם שאליו מנופפים כמחווה לחזור בחזרה
Change:    יד -> רגל
Phonemes:  χ m v χ n m t j j l v ʁ j t j v s f ʁ d ʕ l v l l h v j n v t ʕ v t n f n v f l ʃ l v m v χ f ʁ g l h f v n h l χ j v v n h m n v f f v l ʔ l χ j v v n h ʔ d m ʃ ʔ l j v m n v f f j m χ m χ v v h l χ z v ʁ v χ z ʁ h
Masking: 6.51s - 7.11s

Generating...
Done in 77.3s

Original:



Edited (יד -> רגל):



✅ Done!


In [10]:
print("Starting data preparation...")
print(f"Dataset: {len(ds)} samples")
print(f"Encodec: loaded")
print(f"Phonemizer: working")
print(f"Device: {device}")

# Check what we have
print(f"\nModel vocab has {len(phn2num)} phonemes")
print(f"Our Hebrew phonemes:")
test = hebrew_phonemize("שלום עולם")
for p in test.split():
    status = "✅" if p in phn2num else "❌"
    print(f"  {p}: {status}")

Starting data preparation...
Dataset: 3242 samples
Encodec: loaded
Phonemizer: working
Device: cuda

Model vocab has 80 phonemes
Our Hebrew phonemes:
  ʃ: ✅
  l: ✅
  v: ✅
  m: ✅
  ʕ: ❌
  v: ✅
  l: ✅
  m: ✅


In [11]:
# Check ALL Hebrew phonemes against model vocab
all_hebrew = set()
for i in range(min(100, len(ds))):
    phones = hebrew_phonemize(ds[i]["transcription"])
    for p in phones.split():
        all_hebrew.add(p)

found = []
missing = []
for p in sorted(all_hebrew):
    if p in phn2num:
        found.append(p)
    else:
        missing.append(p)

print(f"Hebrew phonemes in model vocab ({len(found)}): {' '.join(found)}")
print(f"Hebrew phonemes MISSING ({len(missing)}): {' '.join(missing)}")
print(f"\nModel vocab: {sorted(phn2num.keys())}")

Hebrew phonemes in model vocab (14): d f h j k l m n s t v z ʃ ʔ
Hebrew phonemes MISSING (5): g ts ʁ ʕ χ

Model vocab: ['!', ',', '.', '<MUSIC>', '<NOISE>', '<OTHER>', '<SIL>', '?', '_', 'aɪ', 'aɪə', 'aɪɚ', 'aʊ', 'b', 'd', 'dʒ', 'eɪ', 'f', 'h', 'i', 'iə', 'iː', 'iːː', 'j', 'k', 'l', 'm', 'n', 'nʲ', 'oʊ', 'oː', 'oːɹ', 'p', 'r', 's', 't', 'tʃ', 'u', 'uː', 'v', 'w', 'x', 'z', 'æ', 'ææ', 'ç', 'ð', 'ŋ', 'ɐ', 'ɐɐ', 'ɑ', 'ɑː', 'ɑːɹ', 'ɔ', 'ɔɪ', 'ɔː', 'ɔːɹ', 'ə', 'əl', 'ɚ', 'ɛ', 'ɛɹ', 'ɜː', 'ɡ', 'ɡʲ', 'ɪ', 'ɪɹ', 'ɬ', 'ɹ', 'ɾ', 'ʃ', 'ʊ', 'ʊɹ', 'ʌ', 'ʒ', 'ʔ', '̃', '̩', 'θ', 'ᵻ']


In [12]:
# Remap Hebrew phonemes to closest English equivalents
HEBREW_TO_MODEL = {
    'g': 'ɡ',   # same sound, different unicode
    'ts': 'tʃ', # close enough
    'ʁ': 'ɹ',   # both R sounds
    'χ': 'x',   # both velar fricatives
    'ʕ': 'ɑ',   # approximate with open vowel
}

def hebrew_phonemize_mapped(text):
    """Phonemize Hebrew and map to model vocabulary"""
    raw = hebrew_phonemize(text)
    phones = raw.split()
    mapped = []
    for p in phones:
        if p in HEBREW_TO_MODEL:
            mapped.append(HEBREW_TO_MODEL[p])
        else:
            mapped.append(p)
    return ' '.join(mapped)

# Test
test = hebrew_phonemize_mapped("שלום עולם")
print(f"שלום עולם -> {test}")
all_mapped = test.split()
for p in all_mapped:
    status = "✅" if p in phn2num else "❌"
    print(f"  {p}: {status}")

# Verify ALL Hebrew phonemes now map
all_ok = True
for i in range(100):
    phones = hebrew_phonemize_mapped(ds[i]["transcription"]).split()
    for p in phones:
        if p not in phn2num:
            print(f"STILL MISSING: {p}")
            all_ok = False
if all_ok:
    print("\n✅ ALL Hebrew phonemes now map to model vocabulary!")

שלום עולם -> ʃ l v m ɑ v l m
  ʃ: ✅
  l: ✅
  v: ✅
  m: ✅
  ɑ: ✅
  v: ✅
  l: ✅
  m: ✅

✅ ALL Hebrew phonemes now map to model vocabulary!


In [13]:
# Let's try generating speech from Hebrew phonemes using the pretrained model
# This won't sound like Hebrew yet - it's just to verify the phoneme mapping

# Take a short sample
sample = ds[0]
audio_test = np.array(sample["audio"]["array"], dtype=np.float32)
sr_test = sample["audio"]["sampling_rate"]
text_test = sample["transcription"]

print(f"Text: {text_test}")
print(f"Phonemes: {hebrew_phonemize_mapped(text_test)}")
print()

# Let's just play the IPA sounds so you understand the mapping
print("Hebrew letter → IPA → Model phoneme:")
print("  ב (bet)    → v  → v  (same)")
print("  ג (gimel)  → g  → ɡ  (same sound)")
print("  ד (dalet)  → d  → d  (same)")
print("  ה (he)     → h  → h  (same)")
print("  ז (zayin)  → z  → z  (same)")
print("  ח (chet)   → χ  → x  (close - both throat sounds)")
print("  ט (tet)    → t  → t  (same)")
print("  י (yod)    → j  → j  (same)")
print("  כ (kaf)    → χ  → x  (close)")
print("  ל (lamed)  → l  → l  (same)")
print("  מ (mem)    → m  → m  (same)")
print("  נ (nun)    → n  → n  (same)")
print("  ס (samech) → s  → s  (same)")
print("  ע (ayin)   → ʕ  → ɑ  (approximate)")
print("  פ (pe)     → f  → f  (same)")
print("  צ (tsade)  → ts → tʃ (close - both are t+fricative)")
print("  ק (qof)    → k  → k  (same)")
print("  ר (resh)   → ʁ  → ɹ  (both R sounds)")
print("  ש (shin)   → ʃ  → ʃ  (same)")
print("  ת (tav)    → t  → t  (same)")
print()
print("14 out of 19 are EXACT matches")
print("5 are approximations that finetuning will fix")
print()
print("The mapping is good enough - after finetuning the model")
print("will learn the correct Hebrew sounds for each phoneme.")

Text: זו הרכישה הגדולה ביותר בתולדות ebay
Phonemes: z v h ɹ x j ʃ h h ɡ d v l h v j v t ɹ v t v l d v t

Hebrew letter → IPA → Model phoneme:
  ב (bet)    → v  → v  (same)
  ג (gimel)  → g  → ɡ  (same sound)
  ד (dalet)  → d  → d  (same)
  ה (he)     → h  → h  (same)
  ז (zayin)  → z  → z  (same)
  ח (chet)   → χ  → x  (close - both throat sounds)
  ט (tet)    → t  → t  (same)
  י (yod)    → j  → j  (same)
  כ (kaf)    → χ  → x  (close)
  ל (lamed)  → l  → l  (same)
  מ (mem)    → m  → m  (same)
  נ (nun)    → n  → n  (same)
  ס (samech) → s  → s  (same)
  ע (ayin)   → ʕ  → ɑ  (approximate)
  פ (pe)     → f  → f  (same)
  צ (tsade)  → ts → tʃ (close - both are t+fricative)
  ק (qof)    → k  → k  (same)
  ר (resh)   → ʁ  → ɹ  (both R sounds)
  ש (shin)   → ʃ  → ʃ  (same)
  ת (tav)    → t  → t  (same)

14 out of 19 are EXACT matches
5 are approximations that finetuning will fix

The mapping is good enough - after finetuning the model
will learn the correct Hebrew sounds for each phoneme.

In [ ]:
import json, time

# Prepare output directories
os.makedirs("/kaggle/working/training_data/phonemes", exist_ok=True)
os.makedirs("/kaggle/working/training_data/codes", exist_ok=True)
os.makedirs("/kaggle/working/training_data/manifest", exist_ok=True)

print("Processing all samples...")
stime = time.time()

train_entries = []
skipped = 0

for i in range(len(ds)):
    sample = ds[i]
    audio = np.array(sample["audio"]["array"], dtype=np.float32)
    sr = sample["audio"]["sampling_rate"]
    text = sample["transcription"]
    
    if sr != 16000:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    
    duration = len(audio) / 16000
    if duration < 2.0 or duration > 16.0:
        skipped += 1
        continue
    
    phones = hebrew_phonemize_mapped(text)
    if not phones.strip():
        skipped += 1
        continue
    
    utt_id = f"train_{i:06d}"
    
    # Save phonemes
    with open(f"/kaggle/working/training_data/phonemes/{utt_id}.txt", "w") as f:
        f.write(phones)
    
    # Encode audio
    audio_t = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        encoded = encodec_model.encode(audio_t)
        codes = encoded[0][0].cpu()
    
    torch.save(codes, f"/kaggle/working/training_data/codes/{utt_id}.pt")
    
    train_entries.append({
        "utt_id": utt_id,
        "codes": f"/kaggle/working/training_data/codes/{utt_id}.pt",
        "phonemes": f"/kaggle/working/training_data/phonemes/{utt_id}.txt",
        "n_frames": codes.shape[2],
        "text": phones
    })
    
    if i % 300 == 0:
        elapsed = time.time() - stime
        print(f"  {i}/{len(ds)} ({elapsed:.0f}s)")

# Save manifest
with open("/kaggle/working/training_data/manifest/train.txt", "w") as f:
    for e in train_entries:
        f.write(json.dumps(e) + "\n")

elapsed = time.time() - stime
print(f"\nDone in {elapsed:.0f}s")
print(f"Kept: {len(train_entries)}, Skipped: {skipped}")

Processing all samples...
  0/3242 (2s)


In [ ]:
# Process validation set
ds_val = load_dataset("google/fleurs", "he_il", split="validation", trust_remote_code=True)

val_entries = []
skipped = 0
for i in range(len(ds_val)):
    sample = ds_val[i]
    audio = np.array(sample["audio"]["array"], dtype=np.float32)
    sr = sample["audio"]["sampling_rate"]
    text = sample["transcription"]
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    duration = len(audio) / 16000
    if duration < 2.0 or duration > 16.0:
        skipped += 1; continue
    phones = hebrew_phonemize_mapped(text)
    if not phones.strip():
        skipped += 1; continue
    utt_id = f"val_{i:06d}"
    with open(f"/kaggle/working/training_data/phonemes/{utt_id}.txt", "w") as f:
        f.write(phones)
    audio_t = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        codes = encodec_model.encode(audio_t)[0][0].cpu()
    torch.save(codes, f"/kaggle/working/training_data/codes/{utt_id}.pt")
    val_entries.append({"utt_id": utt_id, "codes": f"/kaggle/working/training_data/codes/{utt_id}.pt",
        "phonemes": f"/kaggle/working/training_data/phonemes/{utt_id}.txt",
        "n_frames": codes.shape[2], "text": phones})

with open("/kaggle/working/training_data/manifest/validation.txt", "w") as f:
    for e in val_entries:
        f.write(json.dumps(e) + "\n")

print(f"Validation: {len(val_entries)} kept, {skipped} skipped")

# Save vocab (just use existing phn2num)
with open("/kaggle/working/training_data/vocab.txt", "w") as f:
    for phn, num in sorted(phn2num.items(), key=lambda x: x[1]):
        f.write(f"{phn} {num}\n")

print(f"Vocab: {len(phn2num)} phonemes")
print(f"\nTrain: {len(train_entries)}")
print(f"Val: {len(val_entries)}")
print(f"\n✅ Data ready for training!")

In [ ]:
# Save the pretrained checkpoint with config for training
import copy

# The model is already loaded, save it in the format VoiceCraft training expects
train_ckpt_path = "/kaggle/working/training_data/giga330M_init.pth"
if not os.path.exists(train_ckpt_path):
    torch.save(ckpt, train_ckpt_path)
    print(f"Saved init checkpoint: {train_ckpt_path}")

# Check if VoiceCraft's main.py exists and what it expects
print("\nChecking VoiceCraft training script...")
if os.path.exists("/kaggle/working/VoiceCraft/main.py"):
    with open("/kaggle/working/VoiceCraft/main.py", "r") as f:
        content = f.read()
    print(f"main.py found ({len(content)} chars)")
    # Show the argument parser to see what args it expects
    for line in content.split("\n"):
        if "add_argument" in line and ("manifest" in line or "encodec" in line or "vocab" in line or "resume" in line or "save" in line):
            print(f"  {line.strip()}")
else:
    print("main.py NOT found")
    # List what's available
    for f in os.listdir("/kaggle/working/VoiceCraft"):
        print(f"  {f}")

In [ ]:
# Convert our data to VoiceCraft's expected format
import json

DATA_DIR = "/kaggle/working/training_data"
os.makedirs(f"{DATA_DIR}/encodec_16khz_4codebooks", exist_ok=True)

print("Converting data to VoiceCraft format...")

# Convert train manifest and encodec codes
for split in ["train", "validation"]:
    manifest_lines = []
    with open(f"{DATA_DIR}/manifest/{split}.txt") as f:
        entries = [json.loads(l) for l in f.readlines()]
    
    for entry in entries:
        utt_id = entry["utt_id"]
        n_frames = entry["n_frames"]
        
        # Convert .pt codes to text format (one codebook per line)
        codes = torch.load(entry["codes"], map_location="cpu")  # [1, 4, T]
        codes = codes.squeeze(0)  # [4, T]
        
        enc_txt_path = f"{DATA_DIR}/encodec_16khz_4codebooks/{utt_id}.txt"
        with open(enc_txt_path, "w") as ef:
            for cb in range(codes.shape[0]):
                line = " ".join([str(int(c)) for c in codes[cb]])
                ef.write(line + "\n")
        
        # Manifest format: anything \t utt_id \t length
        manifest_lines.append(f"dummy\t{utt_id}\t{n_frames}")
    
    # Write manifest in their TSV format
    with open(f"{DATA_DIR}/manifest/{split}.txt", "w") as f:
        for line in manifest_lines:
            f.write(line + "\n")
    
    print(f"  {split}: {len(manifest_lines)} samples converted")

# Fix vocab format: their loader expects "num phone" but reads as phn2num = {item[1]:int(item[0])}
# So format is: index phone
with open(f"{DATA_DIR}/vocab.txt", "w") as f:
    for phn, num in sorted(phn2num.items(), key=lambda x: x[1]):
        f.write(f"{num} {phn}\n")

print(f"  Vocab: {len(phn2num)} phonemes")
print("\n✅ Data converted to VoiceCraft format!")

# Verify one sample
test_enc = f"{DATA_DIR}/encodec_16khz_4codebooks/train_000000.txt"
with open(test_enc) as f:
    lines = f.readlines()
print(f"\nVerify encodec file: {len(lines)} codebooks, first line has {len(lines[0].split())} frames")

test_phn = f"{DATA_DIR}/phonemes/train_000000.txt"
with open(test_phn) as f:
    print(f"Verify phonemes: {f.read().strip()}")

In [ ]:
# Move model to CPU, clear GPU completely
model.cpu()
del model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB / {torch.cuda.mem_get_info()[1]/1024**3:.1f} GB")

In [ ]:
import gc

# Delete everything possible
for name in list(globals().keys()):
    if name.startswith('_') or name in ['os', 'sys', 'torch', 'gc', 'np', 'random', 'json', 'device']:
        continue
    try:
        del globals()[name]
    except:
        pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB / {torch.cuda.mem_get_info()[1]/1024**3:.1f} GB")

In [ ]:
import sys, os, gc, argparse, logging, pickle
import torch
import torch.nn as nn
import numpy as np
import random

os.chdir('/kaggle/working/VoiceCraft')
sys.path.insert(0, '/kaggle/working/VoiceCraft')
device = torch.device("cuda")
logging.basicConfig(level=logging.INFO)

# Fix MulticlassAccuracy
class MulticlassAccuracy(nn.Module):
    def __init__(self, *args, **kwargs): super().__init__()
    def forward(self, *args, **kwargs): return torch.tensor(0.0)

import models.voicecraft as vc_module
vc_module.MulticlassAccuracy = MulticlassAccuracy
from models import voicecraft

# Reload model
ckpt = torch.load("./pretrained_models/giga330M.pth", map_location="cpu", weights_only=False)
model_args = ckpt["config"]
phn2num = ckpt["phn2num"]
model = voicecraft.VoiceCraft(model_args)
model.load_state_dict(ckpt["model"])
del ckpt; gc.collect()
model.to(device).train()

print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

# Dataset with short audio to save memory
DATA_DIR = "/kaggle/working/training_data"
EXP_DIR = "/kaggle/working/experiment"
os.makedirs(EXP_DIR, exist_ok=True)

args = argparse.Namespace(
    dataset_dir=DATA_DIR, exp_dir=EXP_DIR,
    manifest_name="manifest", phn_folder_name="phonemes",
    encodec_folder_name="encodec_16khz_4codebooks",
    n_codebooks=4, encodec_sr=50,
    audio_min_length=2, audio_max_length=8,
    text_max_length=200, text_min_length=10,
    drop_long=1, pad_x=0, dynamic_batching=0,
    text_pad_token=model_args.text_pad_token,
    audio_pad_token=model_args.audio_pad_token,
    special_first=0, sep_special_token=False, batch_size=1,
)

from data.gigaspeech import dataset as VCDataset
train_dataset = VCDataset(args, "train")

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=1, shuffle=True,
    num_workers=0, collate_fn=train_dataset.collate, drop_last=True
)

# Test one batch
batch = next(iter(train_loader))
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
print(f"Batch: x={batch['x'].shape}, y={batch['y'].shape}")

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        out = model(batch)

print(f"\nOutput type: {type(out)}")
if isinstance(out, tuple):
    for i, o in enumerate(out):
        if isinstance(o, torch.Tensor):
            print(f"  [{i}] tensor shape={o.shape}")
        elif isinstance(o, list):
            print(f"  [{i}] list len={len(o)}")
        elif isinstance(o, dict):
            print(f"  [{i}] dict keys={o.keys()}")
        else:
            print(f"  [{i}] {type(o)}: {o}")

In [ ]:
print("Output keys and values:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: tensor shape={v.shape}, value={v.item() if v.ndim==0 else 'tensor'}")
    else:
        print(f"  {k}: {type(v)}")

In [2]:
from torch.optim import AdamW

EPOCHS = 10
GRAD_ACCUM = 8
LOG_EVERY = 20
SAVE_EVERY = 500

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
global_step = 0
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

print(f"Training: {len(train_dataset)} samples, {EPOCHS} epochs")
print(f"Batch size=1, grad_accum={GRAD_ACCUM}, effective batch={GRAD_ACCUM}")
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    n_batches = 0
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(train_loader):
        try:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                out = model(batch)
                if out is None:
                    continue
                loss = out['loss'] / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if (batch_idx + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                global_step += 1
                
                if global_step % LOG_EVERY == 0:
                    avg = epoch_loss / n_batches
                    print(f"  Epoch {epoch+1}, Step {global_step}, Loss: {avg:.2f}")
                
                if global_step % SAVE_EVERY == 0:
                    spath = f"/kaggle/working/checkpoints/step_{global_step}.pth"
                    torch.save({"model": model.state_dict(), "config": model_args, 
                               "phn2num": phn2num, "step": global_step}, spath)
                    print(f"  Saved: {spath}")
        
        except Exception as e:
            if "CUDA" in str(e):
                torch.cuda.empty_cache()
            continue
    
    avg_loss = epoch_loss / max(n_batches, 1)
    print(f"\nEpoch {epoch+1}/{EPOCHS} done. Loss: {avg_loss:.2f}, Steps: {global_step}")
    
    spath = f"/kaggle/working/checkpoints/epoch_{epoch+1}.pth"
    torch.save({"model": model.state_dict(), "config": model_args,
               "phn2num": phn2num, "step": global_step}, spath)

print(f"\n✅ Training complete! {global_step} steps")

NameError: name 'model' is not defined